## Imports

In [1]:
from rlnf.reward.reward_model import RewardModel
from rlnf.ppo.critic_network import CriticModel
from rlnf.trainer import RLNFTrainer

import torch
import nemo.collections.asr as nemo_asr
from sentencepiece import SentencePieceProcessor

/home/diarray/office/BambaraASR/bam-asr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load ASR model

In [2]:
asr_model: nemo_asr.models.EncDecCTCModel = nemo_asr.models.EncDecCTCModel.from_pretrained("RobotsMali/stt-bm-quartznet15x5-V0")
asr_model.eval()
asr_model.summarize()

[NeMo W 2025-07-03 19:16:06 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'max_duration': 30, 'min_duration': 0.1, 'batch_size': 64, 'num_workers': 8, 'shuffle': True, 'normalize_transcripts': False, 'labels': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ', "'", '-', 'ŋ', 'ɔ', 'ɛ', 'ɲ', 'ɓ', 'ɾ']}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.
[NeMo W 2025-07-03 19:16:06 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'max_duration': 15, 'batch_size': 32, 'num_workers': 8, 'shuffle': False, 'normalize_transcripts': False, 'labels': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k

[NeMo I 2025-07-03 19:16:06 features:305] PADDING: 16
[NeMo I 2025-07-03 19:16:06 save_restore_connector:275] Model EncDecCTCModel was successfully restored from /home/diarray/.cache/huggingface/hub/models--RobotsMali--stt-bm-quartznet15x5-V0/snapshots/d9f6f590011509b3327992b04564ab8718f526d9/stt-bm-quartznet15x5-V0.nemo.


  | Name              | Type                              | Params | Mode
-------------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0      | eval
1 | encoder           | ConvASREncoder                    | 18.9 M | eval
2 | decoder           | ConvASRDecoder                    | 47.1 K | eval
3 | loss              | CTCLoss                           | 0      | eval
4 | spec_augmentation | SpectrogramAugmentation           | 0      | eval
5 | wer               | WER                               | 0      | eval
-------------------------------------------------------------------------------
18.9 M    Trainable params
0         Non-trainable params
18.9 M    Total params
75.767    Total estimated model params size (MB)
0         Modules in train mode
607       Modules in eval mode

## Configurations

In [3]:
# Load a pre-trained Tokenizer
def load_tokenizer(model_path: str) -> SentencePieceProcessor:
    sp = SentencePieceProcessor()
    sp.Load(model_path)
    return sp

In [4]:
# Load the reward model
reward_model = RewardModel.from_pretrained("/home/diarray/office/BambaraASR/bambara-asr/rlnf/reward/checkpoints/best_model.ckpt")

# Audio preprocessor Config
preprocessor_config = {
    'normalize': 'per_feature',
    'window_size': 0.02,
    'sample_rate': 16000,
    'window_stride': 0.01,
    'window': 'hann',
    'features': 64,
    'n_fft': 512,
    'frame_splicing': 1,
    'dither': 1e-05,
    'stft_conv': False
}

# Training manifest file path
training_manifest = "/home/diarray/office/BambaraASR/media-data/value-dataset/test-rlnf.jsonl"
validation_manifest = "/home/diarray/office/BambaraASR/media-data/value-dataset/test-rlnf-val.jsonl"

# Initialize the critic model
critic_model = CriticModel(n_mel=preprocessor_config['features'])

# Load sentencepiece tokenizer
tokenizer = load_tokenizer("/home/diarray/office/BambaraASR/bambara-asr/bam-tokenizer-spe-bpe-v1024/tokenizer.model")

# Choose the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Loaded RewardModel with config: {
  "n_mel": 64,
  "vocab_size": 1024,
  "embed_dim": 128,
  "lstm_hidden": 128,
  "lstm_layers": 1,
  "audio_conv_channels": 128,
  "audio_conv_layers": 3,
  "head_hidden": 256,
  "dropout": 0.3
}


## Initialize the RLNF trainer

In [5]:
# API KEY: b05ecbe0d101cbf4f7986b5c7e72b3587c06d9ae

trainer = RLNFTrainer(
    reward_model=reward_model,
    critic_model=critic_model,
    asr_model=asr_model,
    train_manifest=training_manifest,
    val_manifest=validation_manifest,
    audio_preprocessor_config=preprocessor_config,
    batch_size=2,
    epochs=3,
    num_workers=0,
    pin_memory=False,
    sp_tokenizer=tokenizer,
    device=device,
    wandb_logging=False,
    run_name="test-run-1"
)

[NeMo I 2025-07-03 19:16:06 features:305] PADDING: 16
[NeMo I 2025-07-03 19:16:06 features:305] PADDING: 16


In [6]:
%cd /home/diarray/office/BambaraASR/media-data/value-dataset/
!ls

/home/diarray/office/BambaraASR/media-data/value-dataset
actor_final.nemo  critic_final.ct  oneliner.jsonl	  test-rlnf.jsonl
audios		  manifest.jsonl   soloni-oneliner.jsonl  test-rlnf-val.jsonl


In [7]:
trainer.train()

Starting epoch: 1 of 3 with config: <wandb.sdk.lib.preinit.PreInitObject object at 0x7f140761c7c0>
Before attemping forward pass, actor and critic are in training mode.


: 